[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [31]:
from os.path import exists
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # router + experts
        self.d_model = d_model
        self.num_experts = num_experts
        self.top_k = top_k

        self.router = nn.Linear(d_model, num_experts)
        # experts
        self.experts = nn.ModuleList(
            [nn.Sequential(
                nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
            ) for _ in range(num_experts)]
        )

    def forward(self, x):
        orig_shape = x.shape
        # route tokens to top-k experts
        if len(x.shape) == 3:
          B, S, d = x.shape
          x_flat = x.reshape(-1, d)
        else:
          x_flat = x
        router_in = self.router(x_flat)
        # print(router_in.shape)
        values, indices = torch.topk(router_in, dim=-1, k=self.top_k)
        weights = torch.softmax(values, dim=-1)
        # print(x_flat.shape, weights.shape)

        moe_out = torch.zeros_like(x_flat)
        for k in range(self.top_k):
          expert_weights = weights[:, k]
          expert_indices = indices[:, k]
          for e in range(len(self.experts)):
            mask = (expert_indices == e)
            if mask.any():
              expert_input = x_flat[mask]
              masked_weights = expert_weights.unsqueeze(-1)[mask]
              # print(masked_weights.shape, expert_input.shape)
              moe_out[mask, :] += masked_weights * self.experts[e](expert_input)
        return moe_out.reshape(orig_shape)


In [32]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [33]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (4.8ms)
  ✅ [2/4] Has router and experts (1.2ms)
  ✅ [3/4] Router logits shape (1.8ms)
  ✅ [4/4] Gradient flow (51.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (58.8ms total)
  Progress saved. Run status() to see your dashboard.

